In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import numpy as np
from loaders._gen_binary import generate_data

In [3]:
import numpy as np
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import balanced_accuracy_score
from sklearn.neighbors import NeighborhoodComponentsAnalysis  # metric learning (linear)

In [4]:
def knn_timeseries_cv(
    X_train, y_train, X_test, y_test,
    *,
    use_metric_learning: bool = True,
    use_dim_reduction: bool = True,
    k_grid = (1, 3, 5),
    n_splits: int = 5,
    random_state: int = 0,
    verbose: int = 0
):
    """
    Returns: dict with best_params, cv_best_score, train_bal_acc, test_bal_acc, best_estimator
    - If use_metric_learning=True, GridSearch will consider both passthrough and NCA before KNN.
    - TimeSeriesSplit keeps temporal order intact.
    """
    # Base pipeline: Standardize -> (optional NCA) -> KNN
    pipe = Pipeline([
        ("scaler", StandardScaler(with_mean=True, with_std=True)),
        ("nca", "passthrough"),  # will be replaced by NCA() in param_grid if enabled
        ("clf", KNeighborsClassifier(metric="minkowski", p=2))
    ])

    n_features = X_train.shape[1]
    if use_dim_reduction:
        nca_components = [2, 4, 8, n_features//2]  # try dimensionalities
    else:
        nca_components = [n_features]

    # Param grid
    # - Always tune k and weights
    param_grid = {
        "clf__n_neighbors": list(k_grid),
        "clf__weights": ["uniform", "distance"],
    }

    if use_metric_learning:
        param_grid = {
            "nca": [NeighborhoodComponentsAnalysis(
                n_components=None, max_iter=200, random_state=random_state
            )],
            "nca__n_components": nca_components,  # try dimensionalities (None = keep all)
            "clf__n_neighbors": list(k_grid),
            "clf__weights": ["uniform", "distance"],
        }

    # TimeSeries CV
    tscv = TimeSeriesSplit(n_splits=n_splits)

    # Grid search with balanced accuracy
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=tscv,
        n_jobs=-1,
        refit=True,
        verbose=verbose
    )

    gs.fit(X_train, y_train)
    best_est = gs.best_estimator_

    # Evaluate on full train/test
    yte_pred = best_est.predict(X_test)

    ba_te = balanced_accuracy_score(y_test,  yte_pred)

    out = {
        "best_params": gs.best_params_,
        "cv_best_score": gs.best_score_,
        "test_bal_acc":  ba_te,
        "best_estimator": best_est,
    }
    return out

# Học khoảng cách + Giảm chiều

In [5]:
bacc = []
for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    res = knn_timeseries_cv(
        X_train, y_train, X_test, y_test,
        use_metric_learning=True,
        use_dim_reduction=True,
    )
    bacc.append(res["test_bal_acc"])

    print(f"Seed {seed}: Test balanced accuracy = {res['test_bal_acc']:.4f}, Best params = {res['best_params']}")

print(f"Balanced accuracy: {np.mean(bacc):.4f} ± {np.std(bacc):.4f}")

Seed 0: Test balanced accuracy = 0.6391, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 12}


Seed 1: Test balanced accuracy = 0.5557, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 2}


Seed 2: Test balanced accuracy = 0.6326, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 12}


Seed 3: Test balanced accuracy = 0.6172, Best params = {'clf__n_neighbors': 3, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 2}


Seed 4: Test balanced accuracy = 0.6290, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 8}


...

Seed 25: Test balanced accuracy = 0.6191, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 12}


Seed 26: Test balanced accuracy = 0.5915, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 8}


Seed 27: Test balanced accuracy = 0.6167, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 8}


Seed 28: Test balanced accuracy = 0.6460, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 4}


Seed 29: Test balanced accuracy = 0.6192, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 4}
Balanced accuracy: 0.6174 ± 0.0249


# Học khoảng cách + Không giảm chiều

In [6]:
bacc = []
for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    res = knn_timeseries_cv(
        X_train, y_train, X_test, y_test,
        use_metric_learning=True,
        use_dim_reduction=False,
    )
    bacc.append(res["test_bal_acc"])

    print(f"Seed {seed}: Test balanced accuracy = {res['test_bal_acc']:.4f}, Best params = {res['best_params']}")

print(f"Balanced accuracy: {np.mean(bacc):.4f} ± {np.std(bacc):.4f}")

Seed 0: Test balanced accuracy = 0.6441, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 1: Test balanced accuracy = 0.5720, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 2: Test balanced accuracy = 0.6124, Best params = {'clf__n_neighbors': 3, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 3: Test balanced accuracy = 0.5797, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 4: Test balanced accuracy = 0.6368, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


...

Seed 25: Test balanced accuracy = 0.6521, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 26: Test balanced accuracy = 0.6037, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 27: Test balanced accuracy = 0.5938, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 28: Test balanced accuracy = 0.6273, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}


Seed 29: Test balanced accuracy = 0.6141, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform', 'nca': NeighborhoodComponentsAnalysis(max_iter=200, random_state=0), 'nca__n_components': 25}
Balanced accuracy: 0.6238 ± 0.0234


# Không học khoảng cách + Không giảm chiều

In [7]:
bacc = []
for seed in range(30):
    pack = generate_data(seed=seed)
    X_train, y_train = pack["train"]
    X_test, y_test = pack["test"]

    res = knn_timeseries_cv(
        X_train, y_train, X_test, y_test,
        use_metric_learning=False,
        use_dim_reduction=False,
    )
    bacc.append(res["test_bal_acc"])

    print(f"Seed {seed}: Test balanced accuracy = {res['test_bal_acc']:.4f}, Best params = {res['best_params']}")

print(f"Balanced accuracy: {np.mean(bacc):.4f} ± {np.std(bacc):.4f}")

Seed 0: Test balanced accuracy = 0.6265, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'distance'}


Seed 1: Test balanced accuracy = 0.6273, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Seed 2: Test balanced accuracy = 0.5938, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 3: Test balanced accuracy = 0.5949, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 4: Test balanced accuracy = 0.6017, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 5: Test balanced accuracy = 0.6193, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


...

Seed 21: Test balanced accuracy = 0.6482, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Seed 22: Test balanced accuracy = 0.5483, Best params = {'clf__n_neighbors': 1, 'clf__weights': 'uniform'}


Seed 23: Test balanced accuracy = 0.6117, Best params = {'clf__n_neighbors': 3, 'clf__weights': 'uniform'}
Seed 24: Test balanced accuracy = 0.6373, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 25: Test balanced accuracy = 0.6067, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Seed 26: Test balanced accuracy = 0.6189, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 27: Test balanced accuracy = 0.5959, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Seed 28: Test balanced accuracy = 0.6364, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}


Seed 29: Test balanced accuracy = 0.6318, Best params = {'clf__n_neighbors': 5, 'clf__weights': 'uniform'}
Balanced accuracy: 0.6214 ± 0.0253
